# KXHIGHNY Event Exploration

Loads the raw event data saved by `data/kalshi.py` (`data/raw/kalshi/kxhighny_events.json`) and does a quick look at what was retrieved. This notebook does not call the Kalshi API itself -- it only reads the already-saved JSON.

In [ ]:
import json
from pathlib import Path

import pandas as pd

EVENTS_PATH = Path("..") / "data" / "raw" / "kalshi" / "kxhighny_events.json"

with open(EVENTS_PATH, "r", encoding="utf-8") as f:
    events = json.load(f)

print(f"Number of events: {len(events)}")

In [ ]:
# Date range covered by the retrieved events (strike_date is null for
# some older events, so we drop those before computing min/max).
strike_dates = [e["strike_date"] for e in events if e.get("strike_date")]
print(f"Date range: {min(strike_dates)}  to  {max(strike_dates)}")

In [ ]:
# Small dataframe: event_ticker, date, title, status.
# Note: the Kalshi Event object has no native "status" field -- only
# nested Market objects do. Status here is derived from the first
# nested market (when Kalshi returned nested markets for that event);
# otherwise it shows as "unknown".

def derive_status(event):
    markets = event.get("markets") or []
    if not markets:
        return "unknown"
    return markets[0].get("status", "unknown")

rows = [
    {
        "event_ticker": e.get("event_ticker"),
        "date": e.get("strike_date"),
        "title": e.get("title"),
        "status": derive_status(e),
    }
    for e in events
]

df = pd.DataFrame(rows)
df.head(10)